## IMPORTS

In [ ]:
from pathlib import Path

import uxarray as ux
import xarray as xr
import numpy as np

## DASK CONFIG

In [ ]:
from dask.distributed import Client

# client = Client(n_workers=1, threads_per_worker=6, processes=False, memory_limit="100GB")
client = Client(n_workers=6, threads_per_worker=2, memory_limit="12GB")
client.dashboard_link

In [ ]:
# Will measure peak notbook (kernel only) memory usage and displaye at the end
from distributed.diagnostics.memory_sampler import MemorySampler
ms = MemorySampler()
cm = ms.sample("full notebook")
cm.__enter__()

In [ ]:
# Will measure peak combined (kernel + dask workers) memory usage and displaye at the end
import os
import threading
import psutil

_peak_rss_bytes = 0
_stop_rss_watch = threading.Event()


def _watch_rss(interval=5):
    global _peak_rss_bytes
    root = psutil.Process(os.getpid())
    while not _stop_rss_watch.is_set():
        try:
            procs = [root] + root.children(recursive=True)
            total = sum(p.memory_info().rss for p in procs if p.is_running())
            _peak_rss_bytes = max(_peak_rss_bytes, total)
        except psutil.Error:
            pass
        _stop_rss_watch.wait(interval)


threading.Thread(target=_watch_rss, daemon=True).start()

In [ ]:
# The distributed.shuffle messages can be too much - suppress them!

import logging
logging.getLogger("distributed.shuffle").setLevel(logging.ERROR)

In [ ]:
TIME_CHUNK = 40
FACE_CHUNK = 2000

## RUN CONFIG (Native Grid)

In [ ]:
# Determine if notebook is to be run on local device or HPC
RUN_ON = 'hpc'  # Notebook to be run on either of "local" | "hpc"

if RUN_ON == 'local':
    KERCHUNK_PATH = "https://data.gdex.ucar.edu/d651007/kerchunk/"
    KERCHUNK_ACCESS = "-remote-osdf"
    KERCHUNK_REMOTE_PROTOCOL = "osdf"
    GRID_FILE = "ne120np4_pentagons_100310.nc"  # Local
elif RUN_ON == 'hpc':
    KERCHUNK_PATH = "/gdex/data/d651007/kerchunk/"
    KERCHUNK_ACCESS = ""  # Kerchunk dataset access, either of "" | "-remote-https" | "-remote-osdf"
    KERCHUNK_REMOTE_PROTOCOL = "file"
    # GRID_FILE = "/glade/p/cesmdata/cseg/inputdata/share/scripgrids/ne120np4_pentagons_100310.nc"  # Glade - this is older?
    GRID_FILE = "/glade/campaign/cesm/cesmdata/cseg/inputdata/share/scripgrids/ne120np4_pentagons_100310.nc"

In [ ]:
MEMBERS = range(2, 11)   # member 1 has no 6-hourly stream

MODEL = "atm"
FREQ = "hour_6"  # or "day_1"
VAR = "PRECT"
VARS = ['TS', 'PSL', 'PRECT']

EXTREME_MEMBER = 10  # to be used for one-member cases

In [ ]:
T_START = "1979-01-01"
T_END = "1988-12-31"

In [ ]:
# Unit conversions

SEC_PER_DAY = 24 * 3600  # 6-hourly output
M_TO_MM = 1000.0

# there are four 6-hour timesteps per day, and exactly 365 days per year in this data;
# it avoids dealing with leap years, and has cftime.DatetimeNoLeap time coordinates.
STEPS_PER_DAY = 4
DAYS_PER_YEAR = 365
STEPS_PER_YEAR = STEPS_PER_DAY * DAYS_PER_YEAR

## UXARRAY (Explore Grid and a single ensemble member)

### Grid

In [ ]:
%%time
uxgrid = ux.open_grid(GRID_FILE)
uxgrid

### Single ensemble member

In [ ]:
# A single ensemble file for sampling
data_filename = "b.e13.BHISTC5.ne120_t12.cesm-ihesp-hires1.0.46-1920-2005." + f"{EXTREME_MEMBER:03d}" + "." + MODEL + "." + FREQ + KERCHUNK_ACCESS + ".parq"
data_file = KERCHUNK_PATH + data_filename

In [ ]:
%%time
# One ensemble member
uxds = ux.open_dataset(
    GRID_FILE, data_file, 
    engine="kerchunk", 
    decode_timedelta=False, 
    storage_options={"remote_protocol": KERCHUNK_REMOTE_PROTOCOL},
    chunks={}
)
uxds

#### Naive plot of a member's PRECT

In [ ]:
%%time
uxds[VAR].isel(time=0).plot()

## YEARLY GLOBAL STATS

In [ ]:
uxds_yearly = uxds[VARS].isel(time=slice(None, None, STEPS_PER_YEAR)).chunk()
print(f'Size: {uxds_yearly.nbytes/1e9:.2f} GB')
uxds_yearly

In [ ]:
%%time 
uxds_yearly = uxds_yearly.load()

In [ ]:
%%time
# naive average: just take mean of all faces across the globe
mean_yearly = uxds_yearly.mean('n_face')

In [ ]:
%%time
# face-area-weighted mean: wmean = ds.weighted_mean()
# (not yet implemented on UxDataset, so can't run that yet.
# workaround: convert to UxDataset, take mean, then convert back.)
weighted_mean_yearly = uxds_yearly.to_array(dim='variable').weighted_mean().to_dataset(dim='variable')

# Also, similar workaround as above: convert to xarray.DataArray
# weighted_mean_yearly = weighted_mean_yearly.to_xarray()
# wmean_jan1 = weighted_mean_yearly  # alias so we can still refer back to this in later sections.

In [ ]:
import matplotlib.pyplot as plt

# visualize the results (pure xarray; nothing in this cell uses uxarray anymore)
fig, axes = plt.subplots(nrows=len(VARS), ncols=1, figsize=(5, 2.5*len(VARS)))
for ax, var in zip(axes, VARS):
    weighted_mean_yearly[var].to_xarray().plot(ax=ax, label="weighted mean", marker='o', ms=2)
    mean_yearly[var].to_xarray().plot(ax=ax, label="mean", ls='--', marker='o', ms=2)
# labeling & formatting
axes[0].legend()
for ax in axes[:-1]:
    ax.set(xlabel="", xticklabels=[])

## WHOLE ENSEMBLE

In [ ]:
# Helper function to open a single ensemble member
def open_member(member):
    hits = sorted(Path(KERCHUNK_PATH).rglob(f"*.{member:03d}.{MODEL}.{FREQ}{KERCHUNK_ACCESS}.parq"))

    if not hits:
        return None
    assert len(hits) == 1, f"expected exactly one file for member {member}, found {len(hits)}: {hits}"

    return ux.open_dataset(
        GRID_FILE, 
        str(hits[0]), 
        engine="kerchunk", 
        decode_timedelta=False, 
        storage_options={"remote_protocol": KERCHUNK_REMOTE_PROTOCOL},
        chunks={"time": TIME_CHUNK}
    )

### Read the ensemble members

In [ ]:
%%time
# Opened lazily only, no eager persist here. Native storage isn't spatially
# chunked (chunks=[1, 777602], one full-globe row per timestep), so every chunk
# read pays a ~33x read/decompress tax before the CONUS mask discards ~97% of it.
members = {m: da for m in MEMBERS if (da := open_member(m)) is not None}

#### CONUS - region to subset

In [ ]:
lat, lon = uxgrid.face_lat.values, uxgrid.face_lon.values
# lon180 = ((lon + 180) % 360) - 180

conus = (lat > 25) & (lat < 50) & (lon > -125) & (lon < -66)

In [ ]:
%%time
members = {key: ds[VAR].sel(time=slice(T_START, T_END)).isel(n_face=conus) for key, ds in members.items()}

In [ ]:
%%time
# Persist just this one member now, staged at the point it's first needed (it's
# reused again below for extreme_metrics) — instead of persisting all 9 members
# upfront, which concentrated the ~33x full-globe read/decompress tax into one
# burst across the whole cluster.
members[EXTREME_MEMBER] = members[EXTREME_MEMBER].persist()

## PRECIPITATION

In [ ]:
print("Source units:", members[EXTREME_MEMBER].attrs.get("units", "unknown"))

# CESM/CAM PRECT is a total precipitation rate in m/s -> convert to mm/day
precip_mm = members[EXTREME_MEMBER] * M_TO_MM * SEC_PER_DAY
precip_mm.attrs["units"] = "mm/day"
precip_mm

In [ ]:
%%time
precip_mm.mean(dim="time").plot(
    cmap="YlGnBu",
    features=["borders", "coastline"],
    title=f"6-hourly PRECT (mm/day) - 10-yr avg — Ensemble member {EXTREME_MEMBER} (USCLIVAR ne120 native grid)" ,
)

## EXTREME PRECIPITATION METRICS

Using the 6-hourly `PRECT` of ensemble members, compute per-gridcell extreme precipitation metrics over CONUS and map their spatial distribution on the native ne120 mesh.

In [ ]:
# Helper function to calculate a couple of preferred percentiles, max, and the total 95th-percentile

def extreme_precip_metrics(uxda, high_q=0.95, extreme_q=0.99, face_chunk=FACE_CHUNK):
    """Per-gridcell extreme precipitation metrics along the "time" dimension."""
    
    uxda = uxda.chunk({"time": -1, "n_face": face_chunk})

    q_high = uxda.quantile(high_q, dim="time", skipna=True).drop_vars("quantile")
    q_extreme = uxda.quantile(extreme_q, dim="time", skipna=True).drop_vars("quantile")
    uxda_max = uxda.max(dim="time", skipna=True)

    above_high_total = uxda.where(uxda >= q_high).sum(dim="time", skipna=True)
    total = uxda.sum(dim="time", skipna=True)
    r95ptot = (above_high_total / total) * 100

    freq_extreme = (uxda >= q_extreme).sum(dim="time") / uxda.sizes["time"] * 100

    metrics = xr.Dataset(
        {
            f"p{int(high_q * 100)}": q_high,
            f"p{int(extreme_q * 100)}": q_extreme,
            "max": uxda_max,
            "r95ptot": r95ptot,
            f"freq_p{int(extreme_q * 100)}": freq_extreme,
        }
    )
    return ux.UxDataset(metrics, uxgrid=uxda.uxgrid)

In [ ]:
%%time
extreme_metrics = extreme_precip_metrics(precip_mm).compute()


### 95th and 99th percentile, max, and r95ptot over 10 years

In [ ]:
extreme_metrics["p95"].plot(
    cmap="YlGnBu",
    features=["borders", "coastline"],
    title=f"6-hourly PRECT (mm/day) - 95th percentile (10 year) — ensemble member {EXTREME_MEMBER} (USCLIVAR ne120 native grid)",
)


In [ ]:
extreme_metrics["p99"].plot(
    cmap="YlGnBu",
    features=["borders", "coastline"],
    title=f"6-hourly PRECT (mm/day) - 99th percentile (10 year) — ensemble member {EXTREME_MEMBER} (USCLIVAR ne120 native grid)",
)


In [ ]:
extreme_metrics["max"].plot(
    cmap="magma",
    features=["borders", "coastline"],
    title=f"6-hourly PRECT (mm/day) - 10-year maximum — ensemble member {EXTREME_MEMBER} (USCLIVAR ne120 native grid)",
)


In [ ]:
extreme_metrics["r95ptot"].plot(
    cmap="YlGnBu",
    features=["borders", "coastline"],
    title=f"6-hourly PRECT (mm/day) - % of total precip from ≥ p95 — ensemble member {EXTREME_MEMBER} (USCLIVAR ne120 grid)",
)


### Ensemble mean of the 99th percentile

In [ ]:
%%time
# Build the p99 metric lazily for every available member, then compute the mean once 
# so dask can schedule the whole ensemble mean as a single parallel graph.

member_p99 = {}
for m, uxda in members.items():
    if m == EXTREME_MEMBER:
        member_p99[m] = extreme_metrics["p99"]
        continue
    precip = (uxda * M_TO_MM * SEC_PER_DAY).chunk({"time": -1, "n_face": FACE_CHUNK})
    member_p99[m] = precip.quantile(0.99, dim="time", skipna=True).drop_vars("quantile")

ensemble_p99 = xr.concat(list(member_p99.values()), dim="member").assign_coords(
    member=list(member_p99.keys())
)

ensemble_p99_mean = ensemble_p99.mean(dim="member").compute()

ensemble_p99_mean_ux = ux.UxDataArray(
    ensemble_p99_mean, uxgrid=members[EXTREME_MEMBER].uxgrid, name="p99_ensemble_mean"
)

In [ ]:
ensemble_p99_mean_ux.plot(
    cmap="YlGnBu",
    features=["borders", "coastline"],
    title="6-hourly PRECT (mm/day) - 99th percentile (10 year) — Ensemble mean (USCLIVAR ne120 native grid)",
)


## NATIVE GRID vs CONSERVATIVELY REMAPPED GRID

To see the effect of conservative regridding, we will now start look ating the conservatively regridded dataset for the workshop as well, and we will take the difference between the results from the native grid and this regridded (lat/lon) grid — this isolates how much conservative aggregation smooths out extremes relative to the native resolution.

### USCLIVAR's regridded data
Read in the regridded CESM-HR global 6-hourly variables used to identify and track extreme weather events (`data_file_regridded_large`) below. The [workshop's website](https://usclivar.org/meetings/high-resolution-modeling-data-access) points to Globus Guest Collections for the datasets, but we will access them through Glade

In [ ]:
# High-frequency (6-hr) outputs for extreme weather event identification and tracking:
data_file_regridded_large = '/glade/campaign/univ/utam0017/Extreme_Weather_Risk_dataset'

# # There was also a smaller dataset intended for the practicum (not used for this notebook):
# data_file_regridded_small = '/glade/campaign/univ/utam0017/High-Resolution_Modeling_Workshop_MESACLIP_dataset'

In [ ]:
import re

_REGRID_MEMBER_TEMPLATE = "b.e13.HF-TNST.rcp85.ne120_t12.1920-2100.{member:03d}"
_REGRID_FILENAME_RE = re.compile(r"\.(\d{10})-(\d{10})\.nc$")


def open_regridded_member(member, lat_slice=None, lon_slice=None):
    """Read one ensemble member's externally-regridded (0.25 deg lat/lon) field."""
    
    member_dir = Path(data_file_regridded_large) / _REGRID_MEMBER_TEMPLATE.format(member=member) / VAR
    hits = sorted(member_dir.glob(f"*.{VAR}.latlon_0.25x0.25_0E.*.nc"),
                  key=lambda p: _REGRID_FILENAME_RE.search(p.name).group(1))
    if not hits:
        return None

    t_start_key, t_end_key = T_START.replace("-", ""), T_END.replace("-", "")
    hits = [
        p for p in hits
        if not (_REGRID_FILENAME_RE.search(p.name).group(2)[:8] < t_start_key
                or _REGRID_FILENAME_RE.search(p.name).group(1)[:8] > t_end_key)
    ]

    ds = xr.open_mfdataset(
        hits,
        combine="nested",
        concat_dim="time",
        decode_times=xr.coders.CFDatetimeCoder(use_cftime=True),
        decode_timedelta=False,
        # Native chunks are [1, 720, 1440] — one full-globe row per timestep, same
        # "not spatially chunked" situation as the kerchunk source above.
        chunks={"time": 40},
        data_vars="minimal", coords="minimal", compat="override",
    )
    
    da = ds[VAR].sel(time=slice(T_START, T_END))
    # lat/lon here are a genuine regular grid, so a contiguous slice() is basic
    # indexing dask can push down cheaply — unlike the CONUS boolean mask on the
    # native ne120 mesh, this doesn't force reading the full globe per timestep.
    if lat_slice is not None:
        da = da.sel(lat=lat_slice)
    if lon_slice is not None:
        da = da.sel(lon=lon_slice)
    return da

In [ ]:
# CONUS bbox in this dataset's 0-360 longitude convention (native mesh used -125/-66)
precip_regridded_raw = open_regridded_member(
    EXTREME_MEMBER, lat_slice=slice(25, 50), lon_slice=slice(235, 294),
)
precip_regridded_mm = precip_regridded_raw * M_TO_MM * SEC_PER_DAY
precip_regridded_mm.attrs["units"] = "mm/day"
precip_regridded_mm

### 99th-percentile (6-hourly PRECT, 10 years) on Regridded Data 

In [ ]:
%%time
p99_regridded = (
    precip_regridded_mm
    .chunk({"time": -1, "lat": 50, "lon": 50})
    .quantile(0.99, dim="time", skipna=True)
    .drop_vars("quantile")
    .compute()
)

In [ ]:
p99_regridded.hvplot(
    cmap="YlGnBu",
    features=["borders", "coastline"],
    title=f"6-hourly PRECT (mm/day) - 99th percentile (10 year) — ensemble member {EXTREME_MEMBER} (USCLIVAR regridded 0.25° grid)",
)

In [ ]:
%%time
p99_native = extreme_metrics["p99"]
p99_native.plot(
    cmap="YlGnBu",
    features=["borders", "coastline"],
    title=f"6-hourly PRECT (mm/day) - 99th percentile (10 year) — ensemble member {EXTREME_MEMBER} (USCLIVAR ne120 native grid)",
)

### Difference

Let's look at the difference of results between the native-mesh `p99_native` read directly by UXarray and the regridded 0.25° grid `p99_regridded`.

In [ ]:
regridded_lon_centers = ((p99_regridded["lon"].values + 180) % 360) - 180  # this product's lon is 0-360
regridded_lat_centers = p99_regridded["lat"].values
uxgrid_from_regridded = ux.Grid.from_structured(lon=regridded_lon_centers, lat=regridded_lat_centers)

p99_native_remapped_to_regridded_grid = p99_native.remap.nearest_neighbor(uxgrid_from_regridded)

In [ ]:
p99_native_remapped_to_regridded_grid.plot(
    cmap="YlGnBu",
    features=["borders", "coastline"],
    title="6-hourly PRECT (mm/day) - 99th percentile (10 year) — native ne120 mesh remapped onto the 0.25° grid",
)

In [ ]:
p99_diff_external = ux.UxDataArray(
    xr.DataArray(
        p99_native_remapped_to_regridded_grid.values - p99_regridded.values.ravel(),
        dims=["n_face"], name="p99_diff_external",
    ),
    uxgrid=uxgrid_from_regridded,
)
p99_diff_external.plot(
    cmap="RdBu_r",
    features=["borders", "coastline"],
    title=(
        "6-hourly PRECT (mm/day) - 99th percentile (10 year) — ensemble member 10\n"
        "USCLIVAR native grid minus USCLIVAR 0.25° regridded grid"
    ),
)

In [ ]:
print("MESACLIP native mesh max p99:                                              ", float(p99_native.max()))
print("MESACLIP native mesh remapped onto MESACLIP regridded 0.25° grid, max p99: ", float(p99_native_remapped_to_regridded_grid.max()))
print("MESACLIP regridded 0.25° grid max p99:                                     ", float(p99_regridded.max()))

In [ ]:
cm.__exit__(None, None, None)
ms.plot()

In [ ]:
_stop_rss_watch.set()
print(f"Peak combined RSS (kernel + dask workers): {_peak_rss_bytes / 1e9:.2f} GB")

In [ ]:
client.close()